# System Setup

## Imports and Installs

In [ ]:
!pip install geopandas
!pip install geojson
!pip install overpass
!pip install alphashape

### Imports

In [ ]:
import pandas as pd
import json
import geojson
#https://github.com/mvexel/overpass-api-python-wrapper/blob/main/README.md
import overpass
#https://pypi.org/project/osm2geojson/
import csv
import os
import logging
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from shapely.geometry import Point, Polygon
import requests
import time
from google.colab import drive
import sys
import ast
from alphashape import alphashape
import geopandas as gpd
import os
from collections import defaultdict
import os.path
from tqdm import tqdm

## Parameters

In [ ]:
state_name = "Ohio"
city_name = "Columbus"
mini = True

## Connecting to Drive

We first mount the drive. This step requires permission to be given. Running the cell should open a permission page to allow colab access to google drive.

In [ ]:
drive.mount('/content/gdrive/', force_remount=True)

Setting the file directory to the data folder. All files will be referenced to this working directory.

Folder structure:

```
📁Drive/
└─ 📁<personal_dir>/
   ├─ 📁<city_name_1> - RL Delivery Data/
   ├─ 📁<city_name_2> - RL Delivery Data/
   └─ 📁Raw GeoJsons/
```


## Assign Data directory path

In [ ]:
personal_dir = "./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/"
data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"

In [ ]:
try:
  os.mkdir(data_dir)
except:
  print(data_dir + " Already Exists")

# See directory content
data = os.listdir(data_dir)
print("Files in directory : ", data)

# Comparison

In [ ]:
import networkx as nx
import pandas as pd
from itertools import combinations
import time

In [ ]:
# === File Paths ===
umst_path = data_dir + "/UMST Graph/graphs/umst_graph.graphml"
hotspot_path = data_dir + "/UMST Graph/graphs/gh_hotspot_graph.graphml"
mst_path = data_dir + "/UMST Graph/graphs/mst_graph.graphml"

# === 1. Load Graphs ===
print("Loading graphs...")
umst_graph = nx.read_graphml(umst_path)
hotspot_graph = nx.read_graphml(hotspot_path)
mst_graph = nx.read_graphml(mst_path)

print(f"Loaded UMST graph: {umst_graph.number_of_nodes()} nodes, {umst_graph.number_of_edges()} edges")
print(f"Loaded Hotspot graph: {hotspot_graph.number_of_nodes()} nodes, {hotspot_graph.number_of_edges()} edges")
print(f"Loaded MST graph: {mst_graph.number_of_nodes()} nodes, {mst_graph.number_of_edges()} edges")

# Average Shortest Distance & Time Calculation

In [ ]:
# === 3. Average Shortest-Path Metric Function ===
def average_shortest_path_metrics(graph):
    nodes = list(graph.nodes())
    distances, times = [], []
    total_pairs = 0

    start_time = time.time()
    for i, (u, v) in enumerate(combinations(nodes, 2)):
        try:
            d = nx.shortest_path_length(graph, source=u, target=v, weight="distance")
            t = nx.shortest_path_length(graph, source=u, target=v, weight="time")
            distances.append(d)
            times.append(t)
        except nx.NetworkXNoPath:
            continue
        total_pairs += 1
        if (i+1) % 1000 == 0:
            print(f"Processed {i+1} pairs...", end="\r")

    avg_distance = sum(distances) / len(distances) if distances else 0
    avg_time = sum(times) / len(times) if times else 0
    elapsed = time.time() - start_time
    print(f"\nComputed {len(distances)} pairs in {elapsed:.2f} sec")

    return {
        "Nodes": len(nodes),
        "Edges": len(graph.edges()),
        "Avg Shortest Distance (m)": avg_distance,
        "Avg Shortest Time (min)": avg_time,
        "Pairs Evaluated": total_pairs
    }


In [ ]:
# === 4. Compare Graphs ===
graphs = {
    "Full_Hotspot": hotspot_graph,
    "MST": mst_graph,
    "UMST": umst_graph
}

results = []
for name, G in graphs.items():
    print(f"\nCalculating metrics for {name} graph...")
    res = average_shortest_path_metrics(G)
    res["Graph"] = name
    results.append(res)

# === 5. Save and Display Summary ===
df = pd.DataFrame(results).set_index("Graph")
df["Edges/Nodes Ratio"] = df["Edges"] / df["Nodes"]
df = df.round(3)

print("\n=== Summary Comparison ===")
print(df)

# Optional: Save to CSV
output_path = data_dir + "/UMST Graph/umst_comparison_summary.csv"
df.to_csv(output_path)
print(f"\nSaved summary to: {output_path}")

## Compare Avg Edge Metrices

In [ ]:
def average_edge_metrics(graph):
    distances = []
    times = []

    nodes = list(graph.nodes())
    num_nodes = len(nodes)
    print(f"Total Edges: {len(graph.edges)}")
    for u, v, data in graph.edges(data=True):
        if "distance" in data:
            distances.append(data["distance"])
        if "time" in data:
            times.append(data["time"])

    avg_distance = sum(distances) / len(distances) if distances else 0
    avg_time = sum(times) / len(times) if times else 0

    return avg_distance, avg_time


In [ ]:
print(f"Full graph")
avg_dist, avg_time = average_edge_metrics(hotspot_graph)
print(f"Average Edge distance: {avg_dist:.2f} km")
print(f"Average Edge time: {avg_time:.2f} mins")

print(f"MST graph")
avg_dist, avg_time = average_edge_metrics(mst_graph)
print(f"Average Edge distance: {avg_dist:.2f} km")
print(f"Average Edge time: {avg_time:.2f} mins")

print(f"UMST graph")
avg_dist, avg_time = average_edge_metrics(umst_graph)
print(f"Average Edge distance: {avg_dist:.2f} km")
print(f"Average Edge time: {avg_time:.2f} mins")